# Setup (Data Preprocessing)

In [26]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("HealthcareETL").getOrCreate()

file_path = "healthcare_dataset.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

df.show()

+-------------------+---+------+----------+-----------------+-----------------+----------------+--------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|               Name|Age|Gender|Blood Type|Medical Condition|Date of Admission|          Doctor|            Hospital|Insurance Provider|    Billing Amount|Room Number|Admission Type|Discharge Date| Medication|Test Results|
+-------------------+---+------+----------+-----------------+-----------------+----------------+--------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|      Bobby JacksOn| 30|  Male|        B-|           Cancer|       2024-01-31|   Matthew Smith|     Sons and Miller|        Blue Cross|18856.281305978155|        328|        Urgent|    2024-02-02|Paracetamol|      Normal|
|       LesLie TErRy| 62|  Male|        A+|          Obesity|       2019-08-20| Samantha Davies|            

In [27]:
total_columns = len(df.columns)
print(f"Total number of columns: {total_columns}")

total_rows = df.count()
print(f"Total number of rows: {total_rows}")

duplicate_count = total_rows - df.dropDuplicates().count()
print(f"Total number of duplicate rows: {duplicate_count}")

Total number of columns: 15
Total number of rows: 55500
Total number of duplicate rows: 534


In [28]:
from pyspark.sql.functions import col, trim, initcap, lower, to_date, when, lit

# Normalize Name
cleaned_df = df.withColumn("Name", initcap(col("Name")))

# Standardize Gender
cleaned_df = cleaned_df.withColumn("Gender", lower(trim(col("Gender"))))

# Convert Dates
cleaned_df = cleaned_df.withColumn("Date of Admission", to_date(col("Date of Admission"), "yyyy-MM-dd"))
cleaned_df = cleaned_df.withColumn("Discharge Date", to_date(col("Discharge Date"), "yyyy-MM-dd"))

# Fill Missing Values
from pyspark.sql.functions import mean

# Fill missing Billing Amount with the median
median_value = cleaned_df.approxQuantile("Billing Amount", [0.5], 0.0)[0]
cleaned_df = cleaned_df.fillna({"Billing Amount": median_value})

# Fill missing categorical values
cleaned_df = cleaned_df.fillna({
    "Medical Condition": "Unknown",
    "Doctor": "Unknown",
    "Hospital": "Unknown",
    "Insurance Provider": "Unknown",
    "Admission Type": "Unknown",
    "Medication": "Unknown",
    "Test Results": "Unknown"
})

In [29]:
# Remove invalid ages
cleaned_df = cleaned_df.filter((col("Age") > 0) & (col("Age") <= 120))

# Remove records with missing Room Number
cleaned_df = cleaned_df.na.drop(subset=["Room Number"])

# Aggregate Billing Amount by Hospital
from pyspark.sql.functions import sum

billing_summary = cleaned_df.groupBy("Hospital").agg(sum("Billing Amount").alias("Total Billing"))
billing_summary.show(truncate=False)

+-------------------------+------------------+
|Hospital                 |Total Billing     |
+-------------------------+------------------+
|Ramirez-Robinson         |96739.00128614204 |
|Foster Lamb, Graham and  |10659.608812944593|
|LLC Massey               |71446.9940966842  |
|Coleman-Aguilar          |34961.90671326528 |
|Group Stein              |25094.95572621151 |
|Smith PLC                |1029424.4491163138|
|Alvarado-Martin          |1077.1696809399066|
|Hall Group               |247201.21558231176|
|and Mayo Chen, Murray    |49400.214028011644|
|Lopez-Wilson             |56481.55593873338 |
|Harris-Farrell           |32166.967383439547|
|Dawson-Williams          |9616.626025733382 |
|Holmes Reed and Johnson, |9492.225487246009 |
|and Lee Rodriguez Morris,|18047.5510046168  |
|Watkins, and Young Perry |41838.22082511615 |
|Nunez-Hamilton           |40446.09173896364 |
|Freeman-Hunter           |29414.68600622586 |
|and Aguilar Sons         |92686.82176385348 |
|Reeves-Edwar

In [30]:
from pyspark.sql.functions import count, col

# Find and show duplicate rows
duplicates_df = df.groupBy(df.columns).agg(count("*").alias("count")).filter(col("count") > 1).drop("count")
duplicates_df.show(truncate=False)

+--------------------+---+------+----------+-----------------+-----------------+----------------+-----------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|Name                |Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor          |Hospital                     |Insurance Provider|Billing Amount    |Room Number|Admission Type|Discharge Date|Medication |Test Results|
+--------------------+---+------+----------+-----------------+-----------------+----------------+-----------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|maLlOry jonEs       |47 |Male  |A-        |Asthma           |2021-09-05       |David Clarke    |Todd and Mullins, Smith      |Aetna             |2692.103462655282 |435        |Emergency     |2021-09-12    |Penicillin |Normal      |
|samUel brYaNt       |53 |Male  |O+        |Arthritis        |2023-0

In [31]:
cleaned_df = cleaned_df.dropDuplicates()

cleaned_df.show(truncate=False)

+------------------+---+------+----------+-----------------+-----------------+-------------------+------------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|Name              |Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor             |Hospital                      |Insurance Provider|Billing Amount    |Room Number|Admission Type|Discharge Date|Medication |Test Results|
+------------------+---+------+----------+-----------------+-----------------+-------------------+------------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|James Ross        |83 |female|A+        |Diabetes         |2024-01-13       |Michael Baker      |Cox-Hester                    |Blue Cross        |10352.20848674086 |394        |Urgent        |2024-01-22    |Aspirin    |Abnormal    |
|Andrea Allen      |47 |female|AB+       |Obesity          |

In [32]:
total_columns = len(cleaned_df.columns)
print(f"Total number of columns: {total_columns}")

total_rows = cleaned_df.count()
print(f"Total number of rows: {total_rows}")

duplicate_count = total_rows - df.dropDuplicates().count()
print(f"Total number of duplicate rows: {duplicate_count}")

Total number of columns: 15
Total number of rows: 54966
Total number of duplicate rows: 0


# Advanced Operations:

In [34]:
cleaned_df.createOrReplaceTempView("healthcare_data")

In [35]:
top_conditions = spark.sql("SELECT `Medical Condition`, COUNT(*) AS Frequency FROM healthcare_data GROUP BY `Medical Condition` ORDER BY Frequency DESC LIMIT 5")
top_conditions.show()
top_conditions.write.csv("Advanced_Operations.csv", header=True, mode="append")

+-----------------+---------+
|Medical Condition|Frequency|
+-----------------+---------+
|        Arthritis|     9218|
|         Diabetes|     9216|
|     Hypertension|     9151|
|          Obesity|     9146|
|           Cancer|     9140|
+-----------------+---------+



In [36]:
biling_amount = spark.sql("SELECT `Medical Condition`, AVG(`Billing Amount`) AS AverageBillingAmount FROM healthcare_data GROUP BY `Medical Condition` ORDER BY AverageBillingAmount DESC;")
biling_amount.show()
biling_amount.write.csv("Advanced_Operations.csv", header=True, mode="append")


+-----------------+--------------------+
|Medical Condition|AverageBillingAmount|
+-----------------+--------------------+
|          Obesity|   25804.36190175903|
|         Diabetes|  25660.478635243962|
|           Asthma|  25633.461696246097|
|        Arthritis|  25511.783246275052|
|     Hypertension|  25503.058720124005|
|           Cancer|  25152.322946627486|
+-----------------+--------------------+



In [37]:
longest_stay = spark.sql("SELECT Name, `Date of Admission`, `Discharge Date`, DATEDIFF(`Discharge Date`, `Date of Admission`) AS LengthOfStay FROM healthcare_data ORDER BY LengthOfStay DESC LIMIT 10;")
longest_stay.show()
longest_stay.write.csv("Advanced_Operations.csv", header=True, mode="append")

+---------------+-----------------+--------------+------------+
|           Name|Date of Admission|Discharge Date|LengthOfStay|
+---------------+-----------------+--------------+------------+
|Michael Oconnor|       2021-09-10|    2021-10-10|          30|
|  Christy Jones|       2020-07-25|    2020-08-24|          30|
|  Raven Gilbert|       2023-04-04|    2023-05-04|          30|
|  Angel Osborne|       2019-07-03|    2019-08-02|          30|
|Shannon Frazier|       2023-03-01|    2023-03-31|          30|
| Brianna Jacobs|       2023-06-15|    2023-07-15|          30|
|   Paula Weaver|       2024-02-22|    2024-03-23|          30|
| Travis Clayton|       2022-07-24|    2022-08-23|          30|
| Holly Andersen|       2022-05-26|    2022-06-25|          30|
|  Hayley Turner|       2023-05-19|    2023-06-18|          30|
+---------------+-----------------+--------------+------------+



In [44]:
number_of_patients = spark.sql("SELECT `Admission Type`, Gender, COUNT(*) AS PatientCount FROM healthcare_data GROUP BY `Admission Type`, Gender ORDER BY `Admission Type`, Gender;")
number_of_patients.show()
number_of_patients.write.csv("Advanced_Operations.csv", header=True, mode="append")

+--------------+------+------------+
|Admission Type|Gender|PatientCount|
+--------------+------+------------+
|      Elective|female|        9281|
|      Elective|  male|        9192|
|     Emergency|female|        9166|
|     Emergency|  male|        8936|
|        Urgent|female|        9023|
|        Urgent|  male|        9368|
+--------------+------+------------+



In [39]:
multiple_admissions = spark.sql("SELECT Name, COUNT(*) AS NumberOfAdmissions FROM healthcare_data GROUP BY Name HAVING COUNT(*) > 1 ORDER BY NumberOfAdmissions DESC;")
multiple_admissions.show()
multiple_admissions.write.csv("Advanced_Operations.csv", header=True, mode="append")

+-----------------+------------------+
|             Name|NumberOfAdmissions|
+-----------------+------------------+
| Michael Williams|                24|
|    Michael Smith|                23|
|     Robert Smith|                21|
|      James Brown|                19|
|      James Smith|                18|
|     John Johnson|                16|
|   Kimberly Smith|                16|
|     James Garcia|                16|
|   James Williams|                16|
|       John Smith|                16|
|   Jennifer Jones|                15|
|    David Johnson|                15|
|    Matthew Smith|                15|
|Christopher Smith|                14|
|    Michael Jones|                14|
|   Jennifer Smith|                14|
|  Robert Williams|                14|
|    Matthew Jones|                14|
|     Thomas Smith|                14|
|    William Smith|                14|
+-----------------+------------------+
only showing top 20 rows



In [40]:
billing_amount_by_age = spark.sql("SELECT CASE WHEN Age BETWEEN 0 AND 18 THEN '0-18' WHEN Age BETWEEN 19 AND 35 THEN '19-35' WHEN Age BETWEEN 36 AND 55 THEN '36-55' ELSE '55+' END AS AgeGroup, AVG(`Billing Amount`) AS AverageBillingAmount FROM healthcare_data GROUP BY AgeGroup ORDER BY AgeGroup;")
billing_amount_by_age.show()
billing_amount_by_age.write.csv("Advanced_Operations.csv", header=True, mode="append")

+--------+--------------------+
|AgeGroup|AverageBillingAmount|
+--------+--------------------+
|    0-18|  26746.284925587155|
|   19-35|   25519.82800406802|
|   36-55|  25461.617977113645|
|     55+|  25569.348115140747|
+--------+--------------------+



In [41]:
number_of_patients = df.groupBy("Hospital").agg(count("Name").alias("Patient Count"))

# Aggregate billing information
billing_info = df.groupBy("Hospital").agg(sum(col("Billing Amount")).alias("Total Billing"))

# Perform an inner join on "Hospital"
joined_df = number_of_patients.join(billing_info, on="Hospital", how="inner")

# Show the result
joined_df.show(10)

# Write joined_df to the same CSV in append mode
joined_df.write.csv("Advanced_Operations.csv", header=True, mode="append")


+--------------------+-------------+------------------+
|            Hospital|Patient Count|     Total Billing|
+--------------------+-------------+------------------+
|    Ramirez-Robinson|            3| 96739.00128614204|
|Foster Lamb, Grah...|            1|10659.608812944593|
|          LLC Massey|            2|  71446.9940966842|
|     Coleman-Aguilar|            1| 34961.90671326528|
|         Group Stein|            2| 25094.95572621151|
|           Smith PLC|           36|1029424.4491163138|
|     Alvarado-Martin|            1|1077.1696809399066|
|          Hall Group|            8|247201.21558231176|
|and Mayo Chen, Mu...|            1|49400.214028011644|
|        Lopez-Wilson|            2| 56481.55593873338|
+--------------------+-------------+------------------+
only showing top 10 rows

